# 08 - Temporal validation per slot (YOLO11n and YOLO11s, fixed model)
Smooths each slot's occupancy over time so a single bad frame does not flip it, **using only the current and earlier frames** (no future leakage), and measures what that costs in delay.

1. **Sequences**: frames of one camera on one day, in time order, split at gaps longer than 3x the camera's usual spacing. MetaPKLot is not video: PKLot frames are about **5 min** apart and CNRPark-EXT about **30 min**, so every delay below is in those units.
2. **Filters** (spec): *consecutive support* (flip only after k agreeing frames), *window* (majority of the last w decisions), *weighted vote* (exponentially weighted slot score, weight alpha on the newest frame), *gated* (a confident score flips the slot at once, a near-threshold one needs 2 agreeing frames), and *none* (per-frame decision, the notebook 07 result). One causal implementation (`src/temporal/filters.py`) is used here and on the Orin.
3. **Metrics**: slot accuracy and macro-F1 per frame; true transitions (a slot's label changes) detected or missed within 3 frames, detection delay in frames and minutes; **false transitions** (the output flips when the truth did not); output flips per slot-hour against true changes per slot-hour.
4. **Choice on val only**, rule fixed before test: among settings within 0.001 val macro-F1 of the best, fewest false transitions per 1000 slot-frames, then shortest delay. Test is reported at that setting next to *none*.

The slot score is the uncalibrated detector score from notebook 07 at each model's val-chosen score threshold. CPU only.

**Inputs (Add Input -> Your Work):** notebook 07 output (`artifacts/occupancy/`: slot scores, occupancy records) and notebook 03 output (`artifacts/splits/` for site, climate and day type).
**Accelerator:** none needed. Internet ON only to clone the repo.

**How to run:** `SMOKE = True` + Run All (about 1 min, a few sequences per split). Then `SMOKE = False` + **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings and inputs
SMOKE = True                      # True: a few sequences per split. False: every val and test sequence
SMOKE_SEQUENCES = 12
VARIANTS = ["yolo11n", "yolo11s"]
TOL_FRAMES = 3                    # a true transition counts as detected if the output follows within this many frames
MAX_GAP_FACTOR = 3.0              # a gap above 3x the camera's median spacing starts a new sequence
F1_TOLERANCE = 0.001              # selection: settings within this of the best val macro-F1 compete on false transitions
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import association as asc
from src.temporal import filters as tf
OCFG = load_yaml("configs/occupancy.yaml")
TCFG = load_yaml("configs/temporal.yaml")

mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(mans[0])}
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())

occ = [p.parent for p in yt.find_files(INPUT_ROOT, "slot_scores_test.csv.gz") if p.parent.name == "occupancy"]
assert occ, "slot_scores_test.csv.gz not found: attach notebook 07's output (full run, not smoke)"
OCC = occ[0]
st07 = json.loads((OCC / "status.json").read_text())
assert st07["state"] == "COMPLETE", "notebook 07 status is {}".format(st07["state"])
THR, REC07 = {}, {}
for v in VARIANTS:
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    assert not rec["smoke_test"], "{}: notebook 07 record is from a smoke run".format(v)
    assert rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], "{}: notebook 07 used a different split".format(v)
    THR[v] = float(rec["chosen_on_val"]["conf_thr"]); REC07[v] = rec
    cfg_thr = (OCFG.get("decision", {}).get("score_threshold") or {}).get(v)
    assert cfg_thr is None or abs(cfg_thr - THR[v]) < 1e-9, "{}: configs/occupancy.yaml threshold differs from notebook 07".format(v)
print("notebook 07 output:", OCC, "| run", st07["run_id"], "| score thresholds (val-chosen in 07):", THR)

def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def load_scores(split):
    t0 = time.time()
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            for v in VARIANTS:
                cols[v].append(float(r["score_" + v]))
    o = {k: np.array(x) for k, x in cols.items()}
    print("{:<5} {} slot rows, {} images, loaded in {:.1f} s".format(split, len(o["gt"]), len(set(cols["image"])), time.time() - t0))
    return o

RAW = {s: load_scores(s) for s in ("val", "test")}
OUT = Path(WORK) / "artifacts" / "temporal"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"temporal": TCFG, "occupancy": OCFG, "smoke": SMOKE, "smoke_sequences": SMOKE_SEQUENCES, "tol": TOL_FRAMES,
                        "gap": MAX_GAP_FACTOR, "f1_tol": F1_TOLERANCE, "thr": THR, "split": SPLIT_INFO["manifest_hash"],
                        "nb07_run": st07["run_id"]})
RUN_ID = new_run_id("tmp")
STATUS = OUT / "status.json"
write_status(STATUS, "08_temporal_validation", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# Sequences: camera + day in time order, split at long gaps; one (frames x slots) matrix per sequence
def build_sequences(o):
    names, first = np.unique(o["image"], return_index=True)
    fr = np.argsort(first); names, first = names[fr], first[fr]             # frames in file order
    idx_of_name = {n: i for i, n in enumerate(names)}
    f_of_row = np.array([idx_of_name[n] for n in o["image"]])
    cam = o["camera"][first]; ts = o["timestamp"][first]; iid = o["image_id"][first]
    sec = np.array([parse_ts(t) for t in ts]); day = np.array([ROW_BY_ID[int(i)]["day"] for i in iid])
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    slots_of_cam = {c: sorted(set(o["slot_id"][o["camera"] == c])) for c in np.unique(cam)}
    seqs = []
    for q in range(sp["seq"].max() + 1 if len(sp["seq"]) else 0):
        frames = np.nonzero(sp["seq"] == q)[0]
        frames = frames[np.argsort(sec[frames], kind="stable")]
        c = cam[frames[0]]
        seqs.append({"frames": frames, "camera": c, "slots": slots_of_cam[c], "sec": sec[frames], "day": day[frames[0]],
                     "spacing_s": sp["spacing_s"][c], "image_ids": iid[frames]})
    # fill matrices
    pos_in_seq = np.zeros(len(names), int); seq_of = sp["seq"]
    for q, s in enumerate(seqs):
        pos_in_seq[s["frames"]] = np.arange(len(s["frames"]))
        T, S = len(s["frames"]), len(s["slots"])
        s["gt"] = np.full((T, S), -1, np.int8); s["obs"] = np.zeros((T, S), bool)
        s["score"] = {v: np.zeros((T, S)) for v in VARIANTS}
        s["col"] = {sid: j for j, sid in enumerate(s["slots"])}
    for r in range(len(o["gt"])):
        f = f_of_row[r]; s = seqs[seq_of[f]]; t = pos_in_seq[f]; j = s["col"][o["slot_id"][r]]
        if o["valid"][r]:
            s["obs"][t, j] = True; s["gt"][t, j] = o["gt"][r]
        for v in VARIANTS:
            s["score"][v][t, j] = o[v][r]
    return seqs, sp["spacing_s"]

SEQ, SPACING = {}, {}
for split in ("val", "test"):
    t0 = time.time()
    SEQ[split], SPACING[split] = build_sequences(RAW[split])
    if SMOKE:
        SEQ[split] = [SEQ[split][i] for i in np.linspace(0, len(SEQ[split]) - 1, min(SMOKE_SEQUENCES, len(SEQ[split]))).astype(int)]
    lens = np.array([len(s["frames"]) for s in SEQ[split]])
    sf = sum(int(s["obs"].sum()) for s in SEQ[split])
    gtt = sum(tf.transition_metrics(s["gt"], s["gt"], s["sec"], 0)["true_transitions"] for s in SEQ[split])
    sh = sum(s["obs"].sum() * s["spacing_s"] / 3600.0 for s in SEQ[split])
    print("{:<5} sequences {:>4} | frames per sequence median {:.0f} (min {}, max {}) | slot-frames {} | true transitions {} ({:.3f} per slot-hour) | {:.0f} s".format(
        split, len(SEQ[split]), np.median(lens), lens.min(), lens.max(), sf, gtt, gtt / sh if sh else 0, time.time() - t0))
print("median frame spacing per camera (val, minutes):", {c: round(x / 60, 1) for c, x in sorted(SPACING["val"].items())})

In [ ]:
# Sweep every temporal setting on VAL, choose by the fixed rule
def evaluate(v, split, cfg, seqs=None):
    seqs = SEQ[split] if seqs is None else seqs
    gts, preds, parts, sf, sh = [], [], [], 0, 0.0
    for s in seqs:
        st = tf.run_sequence(s["score"][v], s["obs"], cfg, THR[v])
        m = s["obs"]
        gts.append(s["gt"][m]); preds.append(st[m])
        parts.append(tf.transition_metrics(s["gt"], np.where(m, st, -1), s["sec"], TOL_FRAMES))
        n = int(m.sum()); sf += n; sh += n * s["spacing_s"] / 3600.0
    c = asc.confusion(np.concatenate(gts) if gts else [], np.concatenate(preds) if preds else [])
    t = tf.merge_transition_metrics(parts, sf, sh)
    row = {"model": v, "split": split, "setting": tf.label(cfg), **{k: cfg.get(k) for k in ("method",) + tf.PARAMS},
           "accuracy": c["accuracy"], "macro_f1": c["macro_f1"], "FV": c["FV"], "FO": c["FO"],
           "false_vacant_rate": c["false_vacant_rate"], "false_occupied_rate": c["false_occupied_rate"]}
    row.update({k: t[k] for k in ("true_transitions", "detected", "missed", "detection_rate", "false_transitions",
                                  "false_per_1000_slot_frames", "pred_transitions_per_slot_hour", "true_transitions_per_slot_hour",
                                  "mean_delay_frames", "p90_delay_frames", "mean_delay_minutes", "slot_frames")})
    return row

GRID = tf.param_grid()
SWEEP, CHOSEN = {}, {}
HDR = "  {:<22} {:>8} {:>8} {:>5} {:>5} {:>9} {:>8} {:>9} {:>8} {:>8}".format(
    "setting", "accuracy", "macroF1", "FV", "FO", "detected", "missed", "false/1k", "delay_fr", "flips/h")
def fmt(r, mark=""):
    return "  {:<22} {:>8} {:>8} {:>5} {:>5} {:>9} {:>8} {:>9} {:>8} {:>8}{}".format(
        r["setting"], r["accuracy"], r["macro_f1"], r["FV"], r["FO"], r["detection_rate"], r["missed"], r["false_per_1000_slot_frames"],
        r["mean_delay_frames"], r["pred_transitions_per_slot_hour"], mark)
try:
    for v in VARIANTS:
        t0 = time.time(); rows = []
        for gi, cfg in enumerate(GRID):
            r = evaluate(v, "val", cfg); r["grid_index"] = gi; rows.append(r)
        SWEEP[v] = rows; CHOSEN[v] = tf.choose(rows, F1_TOLERANCE)
        with open(str(OUT / "val_sweep_{}{}.csv".format(v, SUF)), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
        print("\n{} VAL ({} settings in {:.0f} s; true changes per slot-hour {}):".format(v, len(rows), time.time() - t0, rows[0]["true_transitions_per_slot_hour"]))
        print(HDR)
        for r in rows:
            print(fmt(r, "   <- chosen" if r is CHOSEN[v] else ""))
except Exception as e:
    write_status(STATUS, "08_temporal_validation", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
    raise

In [ ]:
# TEST at the val-chosen setting, next to 'none' (the per-frame notebook 07 decision), overall and per group
def seq_groups(s):
    rows = [ROW_BY_ID[int(i)] for i in s["image_ids"]]
    clim = Counter(r.get("climate") or "unknown" for r in rows).most_common(1)[0][0]
    r0 = rows[0]
    return ["all", "site:" + r0["site"], "camera:" + r0["camera"], "climate:" + clim,
            "day_type:" + ("flagged" if (r0.get("day_flag") or "").strip() else "normal")]

GROUPS = defaultdict(list)
for s in SEQ["test"]:
    for g in seq_groups(s):
        GROUPS[g].append(s)
NONE = {"method": "none"}
RESULTS, GROUP_ROWS = {}, []
for v in VARIANTS:
    ch = {k: CHOSEN[v][k] for k in ("method",) + tf.PARAMS if CHOSEN[v][k] is not None}
    res = {}
    for g, seqs in sorted(GROUPS.items()):
        days = len({(ROW_BY_ID[int(s["image_ids"][0])]["site"], s["day"]) for s in seqs})
        for name, cfg in (("none", NONE), ("chosen", ch)):
            r = evaluate(v, "test", cfg, seqs)
            r.update(group=g, which=name, sequences=len(seqs), days=days, note=de.small_group_note(sum(len(s["frames"]) for s in seqs), days))
            res[(g, name)] = r; GROUP_ROWS.append(r)
    RESULTS[v] = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "model_variant": v,
                  "dataset_split_hash": SPLIT_INFO["manifest_hash"], "notebook07_run_id": st07["run_id"],
                  "score_threshold": THR[v], "score_type": "uncalibrated detector score (score threshold, not a probability)",
                  "chosen_on_val": {k: CHOSEN[v][k] for k in CHOSEN[v] if k not in ("model", "split")},
                  "selection_rule": "max val macro-F1 within {}; then fewest false transitions per 1000 slot-frames; then shortest delay".format(F1_TOLERANCE),
                  "tolerance_frames": TOL_FRAMES, "max_gap_factor": MAX_GAP_FACTOR,
                  "frame_spacing_minutes": {c: round(x / 60, 2) for c, x in SPACING["test"].items()},
                  "test_none": res[("all", "none")], "test_chosen": res[("all", "chosen")],
                  "groups": {"{}|{}".format(g, n): r for (g, n), r in res.items() if g != "all"},
                  "limitation": "frames are 5 min (PKLot) / 30 min (CNRPark-EXT) apart, not video; delays are in those units"}
    (OUT / "temporal_record_{}{}.json".format(v, SUF)).write_text(json.dumps(RESULTS[v], indent=2, default=str))
with open(str(OUT / "groups_test{}.csv".format(SUF)), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(GROUP_ROWS[0])); w.writeheader(); w.writerows(GROUP_ROWS)

print("TEST (slot level; 'none' = per-frame decision of notebook 07; chosen = val-chosen temporal filter)")
for v in VARIANTS:
    print("\n{} (score threshold {}):".format(v, THR[v]))
    print(HDR)
    print(fmt(RESULTS[v]["test_none"], "   none"))
    print(fmt(RESULTS[v]["test_chosen"], "   chosen on val"))
print()
print("{:<8} {:<28} {:<7} {:>4} {:>8} {:>5} {:>5} {:>8} {:>9} {:>8}  {}".format("model", "group", "which", "seqs", "accuracy", "FV", "FO", "missed", "false/1k", "delay_fr", "note"))
for r in GROUP_ROWS:
    if r["group"] != "all":
        print("{:<8} {:<28} {:<7} {:>4} {:>8} {:>5} {:>5} {:>8} {:>9} {:>8}  {}".format(r["model"], r["group"], r["which"], r["sequences"], r["accuracy"],
              r["FV"], r["FO"], r["missed"], r["false_per_1000_slot_frames"], str(r["mean_delay_frames"]), r["note"]))

In [ ]:
# Plots: val trade-off (false transitions vs macro-F1) and one slot's timeline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(VARIANTS), figsize=(6.5 * len(VARIANTS), 4.2), squeeze=False)
for a, v in zip(ax[0], VARIANTS):
    for meth, mk in zip(tf.METHODS, "osD^v"):
        pts = [(r["false_per_1000_slot_frames"], r["macro_f1"], r["setting"]) for r in SWEEP[v] if r["method"] == meth]
        a.scatter([p[0] for p in pts], [p[1] for p in pts], marker=mk, label=meth)
        for x, y, lab in pts:
            a.annotate(lab.replace(meth, "").strip(), (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
    c = CHOSEN[v]; a.scatter([c["false_per_1000_slot_frames"]], [c["macro_f1"]], s=160, facecolors="none", edgecolors="k")
    a.set(xlabel="false transitions per 1000 slot-frames (val)", ylabel="macro-F1 (val)", title="{}: VAL (circle: chosen)".format(v))
    a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(OUT / "val_tradeoff{}.png".format(SUF)), dpi=110); plt.show()

# timeline: the test slot with the most per-frame flips (YOLO11n), raw vs chosen vs truth
v0 = VARIANTS[0]; ch0 = {k: CHOSEN[v0][k] for k in ("method",) + tf.PARAMS if CHOSEN[v0][k] is not None}
best = None
for s in SEQ["test"]:
    raw = tf.run_sequence(s["score"][v0], s["obs"], NONE, THR[v0])
    flips = (np.diff(raw, axis=0) != 0).sum(0)
    j = int(np.argmax(flips))
    if best is None or flips[j] > best[0]:
        best = (int(flips[j]), s, j)
if best:
    _, s, j = best
    st = tf.run_sequence(s["score"][v0], s["obs"], ch0, THR[v0])
    x = (s["sec"] - s["sec"][0]) / 60.0
    fig, a = plt.subplots(figsize=(13, 3.2))
    a.plot(x, s["score"][v0][:, j], color="0.6", lw=1, label="slot score")
    a.axhline(THR[v0], ls=":", color="0.4", lw=0.8)
    a.step(x, s["gt"][:, j] * 1.0 + 0.04, where="post", color="g", label="truth (occupied=1)")
    a.step(x, (s["score"][v0][:, j] >= THR[v0]) * 1.0, where="post", color="orange", lw=1, label="per-frame (none)")
    a.step(x, st[:, j] * 1.0 - 0.04, where="post", color="k", lw=1.5, label=tf.label(ch0))
    a.set(xlabel="minutes from sequence start", title="{} | {} | {} | {}".format(v0, s["camera"], s["day"], s["slots"][j]), ylim=(-0.15, 1.15))
    a.legend(fontsize=8, loc="center right"); plt.tight_layout(); plt.savefig(str(OUT / "slot_timeline{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Finish: environment, status
from src.common.env_record import save_environment
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "08_temporal_validation", RUN_ID, "COMPLETE", CFG_HASH, "smoke={} models={}".format(SMOKE, VARIANTS),
             extra={v: {"chosen_on_val": r["chosen_on_val"]["setting"], "test_none_accuracy": r["test_none"]["accuracy"],
                        "test_chosen_accuracy": r["test_chosen"]["accuracy"],
                        "test_false_per_1000_none": r["test_none"]["false_per_1000_slot_frames"],
                        "test_false_per_1000_chosen": r["test_chosen"]["false_per_1000_slot_frames"]} for v, r in RESULTS.items()})
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it, or paste the output of: the inputs cell, the sequences cell, both VAL tables, the TEST tables, and the two plots.
`artifacts/temporal/` holds the chosen setting per model for notebooks 09-10, so **Save Version** with outputs.